# WindGenerator — Demo

End-to-end wind audio generation using a DDPM diffusion model trained on mel spectrograms.

**Pipeline:** Gaussian noise → DDPM denoising → mel spectrogram → Griffin-Lim → audio

**Runtime:** Go to `Runtime → Change runtime type → T4 GPU` before running.

No setup required — model weights are downloaded automatically from Hugging Face.

## 1. Setup

In [ ]:
!git clone https://github.com/alpercagan/WindGenerator.git /content/WindGenerator 2>/dev/null || \
    (cd /content/WindGenerator && git pull)
!pip install -e /content/WindGenerator -q
!pip install huggingface_hub -q

import torch
print(f"PyTorch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Download model weights from Hugging Face
from huggingface_hub import hf_hub_download
import os

OUTPUT_DIR = '/content/generated'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Downloading checkpoint...")
CHECKPOINT_PATH = hf_hub_download(
    repo_id='alpercagann/wind-generator',
    filename='ckpt_step_074000.pt'
)
print(f"✅ Checkpoint: {CHECKPOINT_PATH}")

MEL_STATS_PATH = hf_hub_download(
    repo_id='alpercagann/wind-generator',
    filename='mel_stats.json'
)
print(f"✅ Mel stats: {MEL_STATS_PATH}")

## 2. Load Model

The diffusion model is a `UNet2DModel` with ~2.5M parameters, trained to denoise mel spectrograms.
Architecture is inferred automatically from the checkpoint.

In [ ]:
import torch, json, sys
sys.path.insert(0, '/content/WindGenerator/src')

from diffusers import UNet2DModel, DDPMScheduler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load checkpoint
ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu')
state_dict = ckpt.get('model', ckpt)

# Infer architecture from checkpoint weights
c0 = state_dict['conv_in.weight'].shape[0]
block_out_channels = (c0, c0*2, c0*4)
has_resnet1 = any('down_blocks.0' in k and 'resnets.1' in k 
                   for k in state_dict.keys())
layers_per_block = 2 if has_resnet1 else 1
n_levels = len([k for k in state_dict if
                k.startswith('down_blocks.') and 
                k.endswith('resnets.0.norm1.weight')])

model = UNet2DModel(
    sample_size=(128, 440),
    in_channels=1, out_channels=1,
    block_out_channels=block_out_channels,
    layers_per_block=layers_per_block,
    down_block_types=tuple(['DownBlock2D'] * n_levels),
    up_block_types=tuple(['UpBlock2D'] * n_levels),
).to(device)

model.load_state_dict(state_dict)
model.disable_gradient_checkpointing()
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {total_params:,} parameters")
print(f"Architecture: channels={block_out_channels}, levels={n_levels}")
print(f"Trained for: {ckpt.get('step', 'unknown')} steps")

with open(MEL_STATS_PATH) as f:
    stats = json.load(f)
mel_mean = stats['mean']
mel_std  = stats['std']
print(f"Mel stats: mean={mel_mean:.4f}, std={mel_std:.4f}")

scheduler = DDPMScheduler(num_train_timesteps=1000)
print("Ready.")

## 3. Generate Audio

**What happens here:**
1. Sample Gaussian noise `(1, 128, 440)` — random starting point
2. DDPM denoising — model iteratively removes noise, converging toward the learned distribution of wind spectrograms
3. Denormalize mel spectrogram
4. `InverseMelScale` — map 128 mel bins back to 513 linear STFT bins
5. Griffin-Lim — estimate phase and reconstruct waveform

In [ ]:
import torchaudio
import soundfile as sf
import IPython.display as ipd
from tqdm.auto import tqdm

def generate_clip(ddpm_steps=100, seed=None):
    if seed is not None:
        torch.manual_seed(seed)

    x = torch.randn(1, 1, 128, 440, device=device)

    scheduler.set_timesteps(ddpm_steps)
    with torch.no_grad():
        for t in tqdm(scheduler.timesteps, desc='Denoising', leave=False):
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                noise_pred = model(x, t).sample
            x = scheduler.step(noise_pred, t, x).prev_sample

    mel = x.squeeze().cpu().clamp(-1, 1)

    # Denormalize
    mel_z      = mel * (4.0 * mel_std) + mel_mean
    mel_linear = (torch.exp(mel_z) - 1e-5).clamp(min=0).unsqueeze(0)

    # Mel → linear spectrogram → waveform
    inverse_mel = torchaudio.transforms.InverseMelScale(
        n_stft=513, n_mels=128, sample_rate=22050,
        f_min=20.0, f_max=11025.0,
    )
    griffin_lim = torchaudio.transforms.GriffinLim(
        n_fft=1024, hop_length=256, win_length=1024,
        power=2.0, n_iter=64,
    )
    segment = griffin_lim(inverse_mel(mel_linear)).squeeze(0)

    # Crossfade with itself for ~10s
    fade_len = 11025
    fade_out = torch.linspace(1, 0, fade_len)
    fade_in  = torch.linspace(0, 1, fade_len)
    seg1, seg2 = segment.clone(), segment.clone()
    seg1[-fade_len:] *= fade_out
    seg2[:fade_len]  *= fade_in
    audio = torch.cat([seg1[:-fade_len], seg1[-fade_len:] + seg2[:fade_len], seg2[fade_len:]])

    peak = audio.abs().max()
    if peak > 0:
        audio = audio / peak * 0.95

    return audio.numpy()

print("Generation function ready.")

In [ ]:
NUM_CLIPS  = 5    # number of clips
DDPM_STEPS = 100  # denoising steps — more = better quality, slower

for i in range(NUM_CLIPS):
    print(f"\n--- Clip {i+1}/{NUM_CLIPS} ---")
    audio = generate_clip(ddpm_steps=DDPM_STEPS, seed=i)
    path = f'{OUTPUT_DIR}/wind_{i+1:02d}.wav'
    sf.write(path, audio, samplerate=22050)
    print(f"Duration: {len(audio)/22050:.1f}s")
    display(ipd.Audio(path))

## 4. Visualize — Generated vs Real Spectrogram

Comparing the structure of a generated mel spectrogram to what the training data looks like.
The similarity in texture shows what the model has learned.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# Generate one mel spectrogram
torch.manual_seed(42)
x = torch.randn(1, 1, 128, 440, device=device)
scheduler.set_timesteps(100)
with torch.no_grad():
    for t in tqdm(scheduler.timesteps, desc='Generating', leave=False):
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            noise_pred = model(x, t).sample
        x = scheduler.step(noise_pred, t, x).prev_sample
generated_mel = x.squeeze().cpu().numpy()

# Load one real mel from dataset inspect outputs
import numpy as np
real_mel_path = '/content/WindGenerator/outputs/inspect/mel_0.png'

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

im0 = axes[0].imshow(generated_mel, aspect='auto', origin='lower',
                      cmap='magma', vmin=-1, vmax=1)
axes[0].set_title('Generated (diffusion model output)', fontsize=11)
axes[0].set_xlabel('Time frames (440 = 5.12s)')
axes[0].set_ylabel('Mel frequency bins (128)')
plt.colorbar(im0, ax=axes[0], label='Normalized value')

# Show a real mel from inspect folder if available
if os.path.exists(real_mel_path):
    img = plt.imread(real_mel_path)
    axes[1].imshow(img, aspect='auto')
    axes[1].set_title('Real wind recording (from dataset)', fontsize=11)
else:
    axes[1].text(0.5, 0.5, 'Real mel not available',
                ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title('Real wind recording', fontsize=11)

plt.tight_layout()
plt.show()

## 5. Download Generated Clips

In [ ]:
from google.colab import files
import glob

wav_files = sorted(glob.glob(f'{OUTPUT_DIR}/*.wav'))
print(f"{len(wav_files)} clips generated:")
for f in wav_files:
    print(f"  {os.path.basename(f)} ({os.path.getsize(f)/1024:.0f} KB)")

for f in wav_files:
    files.download(f)

---
## Notes

**On output quality:** Some clips sound more wind-like than others — this is expected. The diffusion model samples stochastically from a learned distribution; some samples are closer to the training data manifold than others. The model was trained for 74,000 steps (target: 100,000).

**On the metallic quality:** This comes from Griffin-Lim phase reconstruction, not from the diffusion model. Griffin-Lim estimates phase iteratively but the solution is not perceptually optimal. The learned spectrogram content (frequency distribution, temporal texture) is separate from this artifact.

**On `ddpm_steps`:** More steps = better quality, slower. 50 is fast; 100–200 is better. Diminishing returns above 200.

**Repository:** [github.com/alpercagan/WindGenerator](https://github.com/alpercagan/WindGenerator)  
**Model weights:** [huggingface.co/alpercagann/wind-generator](https://huggingface.co/alpercagann/wind-generator)